# Isooctane forcefield

This section describes the development and analysis of a method to improve the surface tension of isooctane by adjusting the epsilon parameter within the gromos54a7 force field.

In [2]:
# extracted from the gromos54a7 force field itp file.
# 
#"gromos54a7 = "../data/itps/ffnonbonded_gromos54a7_atb_original.itp"
#
#

original_isoo_itp = """
[ atomtypes ]
;name  at.num   mass      charge  ptype       c6           c12
  CH0    6      0.000      0.000     A  0.0023970816  0.0002053489
  CH1    6      0.000      0.000     A  0.00606841  9.70225e-05
  CH2    6      0.000      0.000     A  0.0074684164  3.3965584e-05
  CH3    6      0.000      0.000     A  0.0096138025  2.6646244e-05
"""

In [3]:
print(original_isoo_itp)


[ atomtypes ]
;name  at.num   mass      charge  ptype       c6           c12
  CH0    6      0.000      0.000     A  0.0023970816  0.0002053489
  CH1    6      0.000      0.000     A  0.00606841  9.70225e-05
  CH2    6      0.000      0.000     A  0.0074684164  3.3965584e-05
  CH3    6      0.000      0.000     A  0.0096138025  2.6646244e-05



# Epsilon parameter adjustment for isooctane

The force field for isooctane was modified by changing the epsilon parameter $\epsilon_{ij}$ of the Lennard-Jones potential, which determines the strength of the van der Waals interactions between atoms.


$$
V_{LJ}\left(\boldsymbol{r}_{ij}\right)=4\epsilon_{ij}\left[\left(\cfrac{\sigma_{ij}}{r_{ij}}\right)^{12}-\left(\cfrac{\sigma_{ij}}{r_{ij}}\right)^{6}\right]
$$



The Lennard-Jones potential parameters in the gromos54a7 force field are not directly defined by this parameter. Instead, they are represented in a different way, which is described below:

$$
V_{LJ}\left(\boldsymbol{r}_{ij}\right)=\cfrac{C_{ij}^{\left(12\right)}}{r_{ij}^{12}}-\cfrac{C_{ij}^{\left(6\right)}}{r_{ij}^{6}}
$$

The equations below demonstrate the relationship between the parameters sigma, epsilon, A, and B, which allows for conversion between the two ways of representing the Lennard-Jones potential:

$$
C^{\left(12\right)}=4\epsilon\sigma^{12}
$$

$$
C^{\left(6\right)}=4\epsilon\sigma^{6}
$$

As the epsilon parameter is modified, it's essential to adjust c6 and c12 accordingly, as they represent the strength of the attractive and repulsive forces in the Lennard-Jones potential. The script below provides a practical example of how to modify c6 and c12 while varying epsilon, using the original values of c6 and c12 from the gromos54a7 force field for the isooctane.

In [32]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict

def fsigma(c6, c12):
    return np.power((c12/c6), 1/6)

def feps(c6, c12):
    return c6*c6/(4*c12)

def fc6(sigma, eps):
    return 4*eps*np.power(sigma, 6)

def fc12(sigma, eps):
    return 4*eps*np.power(sigma, 12)

# extracted from the gromos54a7 force field itp file.
# "gromos54a7 = "../data/itps/ffnonbonded_gromos54a7_atb_original.itp"

original_isoo_itp = """
[ atomtypes ]
;name  at.num   mass      charge  ptype       c6           c12
  CH0    6      0.000      0.000     A  0.0023970816  0.0002053489
  CH1    6      0.000      0.000     A  0.00606841  9.70225e-05
  CH2    6      0.000      0.000     A  0.0074684164  3.3965584e-05
  CH3    6      0.000      0.000     A  0.0096138025  2.6646244e-05
""".split("\n")

dic = defaultdict(dict)

for idx, line in enumerate(original_isoo_itp[3:-1]):

    sp = line.split()
    dic['name'][idx] = sp[0]
    dic['at.num'][idx] = int(sp[1])
    dic['mass'][idx] = float(sp[2])
    dic['charge'][idx] = float(sp[3])
    dic['ptype'][idx] = sp[4]
    dic['c6'][idx] = float(sp[5])
    dic['c12'][idx] = float(sp[6])

df_54a7 = pd.DataFrame(dic)

ks = [0.875, 0.9  , 0.925, 0.95]
for k in ks:

    print(f"c6 and c12 values for k={k} ==> k*epsilon={mod_epsilon}:\n")
    print(f";name  at.num   mass      charge  ptype       c6           c12")
    
    
    for idx in df_54a7.index:
    
        c6 = df_54a7["c6"].loc[idx]
        c12 = df_54a7["c12"].loc[idx]
        
        epsilon = feps(c6, c12)
        sigma = fsigma(c6, c12)
        
        mod_epsilon = k*epsilon
    
        mod_c6 = fc6(sigma, mod_epsilon)
        mod_c12 = fc12(sigma, mod_epsilon)
    
        name = df_54a7["name"].loc[idx]
        atnum = df_54a7['at.num'].loc[idx]
        mass = df_54a7['mass'].loc[idx]
        charge = df_54a7['charge'].loc[idx]
        ptype = df_54a7['ptype'].loc[idx]
      
        print(f"{name:>5}{atnum:>5d}{mass:>11.3f}{charge:>11.3f}{ptype:>6}{mod_c6:>14.10f}{mod_c12:>15.7e}")
    
    print(40*"--")



c6 and c12 values for k=0.875 ==> k*epsilon=0.8237928259565956:

;name  at.num   mass      charge  ptype       c6           c12
  CH0    6      0.000      0.000     A  0.0020974464  1.7968029e-04
  CH1    6      0.000      0.000     A  0.0053098588  8.4894688e-05
  CH2    6      0.000      0.000     A  0.0065348644  2.9719886e-05
  CH3    6      0.000      0.000     A  0.0084120772  2.3315464e-05
--------------------------------------------------------------------------------
c6 and c12 values for k=0.9 ==> k*epsilon=0.7587565502231802:

;name  at.num   mass      charge  ptype       c6           c12
  CH0    6      0.000      0.000     A  0.0021573734  1.8481401e-04
  CH1    6      0.000      0.000     A  0.0054615690  8.7320250e-05
  CH2    6      0.000      0.000     A  0.0067215748  3.0569026e-05
  CH3    6      0.000      0.000     A  0.0086524222  2.3981620e-05
--------------------------------------------------------------------------------
c6 and c12 values for k=0.925 ==> k*epsi

In [1]:
import sys
sys.path.append('../code/')
import getpass
from herramientas import *
from isoo import *
from system_creator import *
from topoles import *
from groFile import *


t0_grl = time.time()
user = getpass.getuser()

herr = herramientas()
iso = isoo()
tp = topoles()
call_grompp = call_grompp_conf()

# gmx = '/usr/local/gromacs/bin/gmx'
gmx = "/usr/local/apps/gromacs-2020-beta1/build/bin/gmx"
gmx = "/home/antonio/gmx-2022.4/bin/gmx"
gmx = "/home/antadlp/gmx_2021_7/bin/gmx"


NUM_PROC = 6

PATH_GRL_0_0 = "/disco2/SIMS"

NAME_CARPET0 = "isoos_6_jul_2024_test1"

dP_LVL0 = iso.set_paths_lvl0(path_grl_0_0 = PATH_GRL_0_0,
                                     name_carpet0=NAME_CARPET0)
# =====================
# values to productions runs
# num_mols = 5000

# nsteps_min1 = 10000
# nsteps_min2 = 10000
# nsteps_nvt1 = 2500000
# nsteps_npt1 = 2500000
# nsteps_nvt2 = 2500000

# nstxout = 5000
# nstvout = nstxout
# nstenergy = nstxout
# nstlog = nstxout

# =====================
# values to TEST runs
num_mols = 100

nsteps_min1 = 1000
nsteps_min2 = 1000
nsteps_nvt1 = 25000
nsteps_npt1 = 25000
nsteps_nvt2 = 25000

nstxout = 500
nstvout = nstxout
nstenergy = nstxout
nstlog = nstxout

# =====================

dt = 0.025
# eps_factors = np.arange(0.875, 1.2 + dt, dt)
# array([0.875, 0.9  , 0.925, 0.95 , 0.975, 1.   , 1.025, 1.05 , 1.075,
#        1.1  , 1.125, 1.15 , 1.175, 1.2  ])
# eps_factors = [0.975, 1. , 1.025, 1.05 , 1.075, 1.1  , 1.125, 1.15 , 1.175, 1.2]
# eps_factors = [1.1  , 1.125, 1.15 , 1.175, 1.2]

eps_factors = [0.875, 0.9  , 0.925, 0.95 , 0.975, 1.   , 1.025, 1.05 , 1.075,  1.1  , 1.125, 1.15 , 1.175, 1.2  ]

# ks = [0.88, 0.9, 0.92, 0.95]


de = 5
NNVT2 = 5

dic_dens = {}
dic_dens['density'] = {}
dic_dens['epsilon'] = {}

dic_surfT = {}
dic_surfT['epsilon'] = {}
dic_surfT['surfT'] = {}

din_counter = 0
for _eps_factor in eps_factors:
    
    t1_grl = time.time()
    
    eps_factor = np.round(_eps_factor, decimals=2)
     
    NAME_DIN = 'din_eps_' + str(eps_factor)
    
    print("\n\n !==============")
    print("{}".format(NAME_DIN))
    
    dP_LVL1=iso.set_paths_lvl1(name_din=NAME_DIN, dP_LVL0=dP_LVL0)
    
    PATH_CELDA0 = sys_isooctano(num_mols=num_mols, dP_LVL1=dP_LVL1, gmx=gmx, de=de, lado_celda0=5)

    PATH_FFNB = tp.get_ffnb_g016_atypes(path_grl1=dP_LVL1['GRL1'],
                                 name_din=NAME_DIN,
                                 epsilon_factor=_eps_factor)
    
    PATH_TOPOL = tp.wrapper_create_topol_g016(path_grl1=dP_LVL1['GRL1'],
                                              path_celda0=PATH_CELDA0,
                                              path_ffnb_g016=PATH_FFNB,
                                              name_din=NAME_DIN)


    to = time.time()
    iso.wrapper_min1(path_celda0=PATH_CELDA0,
                     dP1=dP_LVL1,
                     num_proc=NUM_PROC,
                     gmx=gmx,
                     nsteps=nsteps_min1,
                     path_topol=PATH_TOPOL)
    
    tmin1 = time.time() - to
    print("MIN 1: {}".format(tmin1))    

    
    iso.wrapper_min2(dP1=dP_LVL1,
                     num_proc=NUM_PROC,
                     gmx=gmx,
                     nsteps=nsteps_min2,
                     path_topol=PATH_TOPOL)    
    tmin2 = time.time() - to
    print("MIN 2: {}".format(tmin2))        
    

    iso.wrapper_nvt1(dP1=dP_LVL1,
                     num_proc=NUM_PROC,
                     gmx=gmx,
                     nsteps=nsteps_nvt1,
                     nstxout=nstxout,
                     nstvout=nstvout,
                     nstenergy=nstenergy,
                     nstlog=nstlog,
                     path_topol=PATH_TOPOL)    
    
    tnvt1 = time.time() - to
    print("NVT 1: {}".format(tnvt1))  


    iso.wrapper_npt1(dP1=dP_LVL1,
                     num_proc=NUM_PROC,
                     gmx=gmx,
                     nsteps=nsteps_npt1,
                     nstxout=nstxout,
                     nstvout=nstvout,
                     nstenergy=nstenergy,
                     nstlog=nstlog,
                     path_topol=PATH_TOPOL)    
    
    
    tnpt1 = time.time() - to
    print("NPT 1: {}".format(tnpt1))
    

    df_nvt2 = call_grompp.call_estandar_nvt_000(nsteps=nsteps_nvt2,
                                                dt=0.002,
                                                nstxout=nstxout,
                                                nstvout=nstvout,
                                                nstenergy=nstenergy,
                                                nstlog=nstlog)
    
    for i in range(NNVT2):
        
        iso.wrapper_nvt2s(name_din=NAME_DIN,
                           dP1=dP_LVL1,
		           path_topol=PATH_TOPOL,
                           num_proc=NUM_PROC,
                           df_nvt2=df_nvt2, gmx=gmx, i=i)
         
         

     



 !==============
din_eps_0.88


                 :-) GROMACS - gmx insert-molecules, 2021.7 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            Tee

MIN 1: 2.537248373031616



Back Off! I just backed up /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/min2/min2.trr to /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/min2/#min2.trr.1#

Back Off! I just backed up /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/min2/min2.edr to /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/min2/#min2.edr.1#

Steepest Descents:
   Tolerance (Fmax)   =  1.00000e+01
   Number of steps    =         1000

Energy minimization reached the maximum number of steps before the forces
reached the requested precision Fmax < 10.

writing lowest energy coordinates.

Back Off! I just backed up /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/min2/min2.gro to /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/min2/#min2.gro.1#

Steepest Descents did not converge to Fmax < 10 in 1001 steps.
Potential Energy  =  2.7824067e+03
Maximum force     =  6.4821266e+01 on atom 654
Norm of force     =  9.0317748e+00

GROMACS reminds you: "A robot will be truly autonomous when you instruct it to go to wo

MIN 2: 5.019953489303589


...

                      :-) GROMACS - gmx grompp, 2021.7 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            Teemu Mu

nvt2_3: 2.3101093769073486


                      :-) GROMACS - gmx grompp, 2021.7 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen          M. Eric Irrgang      
      Aleksei Iupinov           Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            Teemu Mu

nvt2_4: 2.09464693069458



Writing final coordinates.

               Core t (s)   Wall t (s)        (%)
       Time:       10.564        1.761      600.0
                 (ns/day)    (hour/ns)
Performance:     2453.574        0.010

GROMACS reminds you: "I am driven by two main philosophies: know more today about the world than I knew yesterday and lessen the suffering of others. You'd be surprised how far that gets you." (Neil deGrasse Tyson)

                      :-) GROMACS - gmx energy, 2021.7 (-:

                            GROMACS is written by:
     Andrey Alekseenko              Emile Apol              Rossen Apostolov     
         Paul Bauer           Herman J.C. Berendsen           Par Bjelkmar       
       Christian Blau           Viacheslav Bolnykh             Kevin Boyd        
     Aldert van Buuren           Rudi van Drunen             Anton Feenstra      
    Gilles Gouaillardet             Alan Gray               Gerrit Groenhof      
       Anca Hamuraru            Vincent Hindriksen     

In [2]:
' '.join(['/home/antadlp/gmx_2021_7/bin/gmx', 'grompp', '-f', '/disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/nvt2_0/nvt2_0.mdp', '-p', '/disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/topol.top', '-c', '/disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/npt1/nvt2_in.gro', '-o', '/disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/nvt2_0/nvt2_0.tpr', '-pp', '/disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/nvt2_0/nvt2_0.top', '-po', '/disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/nvt2_0/nvt2_0_dump.mdp', '-maxwarn', '10'])

'/home/antadlp/gmx_2021_7/bin/gmx grompp -f /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/nvt2_0/nvt2_0.mdp -p /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/topol.top -c /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/npt1/nvt2_in.gro -o /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/nvt2_0/nvt2_0.tpr -pp /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/nvt2_0/nvt2_0.top -po /disco2/SIMS/isoos_6_jul_2024_test1/din_eps_0.88/nvt2_0/nvt2_0_dump.mdp -maxwarn 10'